In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps
from conus_biomass.train_models.train_all_models import load_data

# Figure 1a

In [ ]:
fia_data = load_data()

In [ ]:
num_fires = fia_data["fires_occurred"].sum(dim="plotid")
num_measured_plots = (fia_data["measyear_2"] > fia_data["year"]).sum(dim="plotid")
fia_frac_plots = (num_fires * 100 / num_measured_plots).where(num_measured_plots > 10000)
fia_frac_plots = fia_frac_plots.where(fia_frac_plots["year"] >= 1990)

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")
inputs = inputs.rio.write_crs(crs)

In [ ]:
western_states = maps.SHP_WESTERN.to_crs(crs)

In [ ]:
is_forest = xr.open_dataset(
    dir_info.dir_processed + "data_on_ref_grid/1000m/" + "is_forest_tseries.nc"
)
is_forest = is_forest.rio.write_crs(crs)

In [ ]:
is_forest_clipped = is_forest["is_forest"].rio.clip(western_states.geometry)

# Temporary fix because of error in 2008
# is_forest_clipped.loc[dict(year=2008)] = is_forest_clipped.sel(year=2009).values

In [ ]:
years_after_fire_clipped = inputs["years_after_fire"].rio.clip(western_states.geometry)

In [ ]:
burned_forest = (years_after_fire_clipped == 0) * (is_forest_clipped / 100)
recently_burned_forest = (years_after_fire_clipped <= 10) * (is_forest_clipped / 100)

In [ ]:
burned_forest_tseries = burned_forest.sum(dim=["y", "x"])[:-1]
recently_burned_forest_tseries = recently_burned_forest.sum(dim=["y", "x"])[:-1]
forest_area = (is_forest_clipped / 100).sum(dim=["y", "x"])[:-1]


fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10))

yvals = burned_forest_tseries * 100 / forest_area
plt.subplot(2, 2, 1)
ax = axes[0, 0]
plt.plot(yvals["year"], yvals, "-", color="#690F0F", linewidth=3, markersize=15, label="MTBS")
plt.plot(
    yvals["year"],
    yvals.rolling(year=10, center=True).mean(),
    "--",
    color="#690F0F",
    linewidth=1,
    label="MTBS (centered 10-year rolling mean)",
)
plt.axhline(y=0, linestyle=":", color="k")
plt.ylabel("Percent")
ax.spines[["right", "top"]].set_visible(False)
ax.set_title("Forest area burned annually")
plt.plot(fia_frac_plots["year"], fia_frac_plots, color="salmon", linewidth=3, label="FIA")
# plt.plot(fia_frac_plots['year'], fia_frac_plots.rolling(year=10,center=True).mean(), '--',color='salmon', linewidth=1, label='FIA (centered 10-year rolling mean)')
plt.legend()

yvals = recently_burned_forest_tseries * 100 / forest_area
plt.subplot(2, 2, 2)
ax = axes[0, 1]
plt.plot(yvals["year"], yvals, ".-", color="#690F0F", linewidth=3, markersize=15)
plt.axhline(y=0, linestyle=":", color="k")
plt.ylabel("Percent")
plt.xlim([1999, 2022])
plt.ylim([-0.3, 10])
ax.spines[["right", "top"]].set_visible(False)
ax.set_title("Forest area burned within past 10 years")

yvals = burned_forest_tseries * 100 / 1e6
plt.subplot(2, 2, 3)
ax = axes[1, 0]
plt.plot(yvals["year"], yvals, ".-", color="#690F0F", linewidth=3, markersize=15, label="MTBS")
plt.plot(
    yvals["year"], yvals.rolling(year=10, center=True).mean(), "--", color="#690F0F", linewidth=1
)
plt.axhline(y=0, linestyle=":", color="k")
plt.ylabel("Million ha")
plt.legend()
ax.spines[["right", "top"]].set_visible(False)

yvals = recently_burned_forest_tseries * 100 / 1e6
plt.subplot(2, 2, 4)
ax = axes[1, 1]
plt.plot(yvals["year"], yvals, ".-", color="#690F0F", linewidth=3, markersize=15)
plt.axhline(y=0, linestyle=":", color="k")
plt.xlim([1999, 2022])
plt.ylabel("Million ha")
ax.spines[["right", "top"]].set_visible(False)

Fires are also missing from the MTBS dataset owing to gaps in the Landsat archive (J. Lecker, pers. comm.). Gaps are particularly evident during the Landsat ‘commercialisation’ period, 1985–99, when the acquisition and archiving of images was largely driven by market demand (Tucker et al. 2004). "Sources and implications of bias and uncertainty in a century of US wildfire activity data"

# Figure 1b

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")

In [ ]:
is_forest = inputs["is_forest"].sel(year=2021)

In [ ]:
years_after_fire = inputs["years_after_fire"].sel(year=2022)

In [ ]:
fire_year = (2022 - years_after_fire).where(years_after_fire < 80)

In [ ]:
western_states = maps.SHP_WESTERN.to_crs(crs)

In [ ]:
mapdata = fire_year.where(is_forest)
mapdata = mapdata.rio.write_crs(crs)
mapdata_clipped = mapdata.rio.clip(western_states.geometry)

In [ ]:
is_forest = is_forest.rio.write_crs(crs)
is_forest_clipped = is_forest.rio.clip(western_states.geometry)

In [ ]:
# Define the boundaries for your discrete bins
boundaries = [2000, 2005, 2010, 2020, 2025]

# Create the continuous colormap
cmap = mcolors.LinearSegmentedColormap.from_list("my_cmap", ["#FBBBA8", "#690F0F"])

n_colors = 5
discrete_cmap = mcolors.ListedColormap(cmap(np.linspace(0, 1, n_colors)))

In [ ]:
ax = maps.plot_map(
    is_forest_clipped,
    shp=western_states,
    latlon=False,
    cbar_label="Percent Forest",
    cmap=plt.cm.binary,
    add_colorbar=False,
    clims=[0, 800],
    savefig=None,
)

maps.plot_map(
    mapdata_clipped,
    shp=western_states,
    latlon=False,
    title="Year of most recent fire",
    cbar_label="Year of most recent fire",
    cmap=discrete_cmap,
    savefig=None,
    clims=[2000, 2025],
    add_colorbar=True,
    ax=ax,
)

# Figure 1 (all panels)

In [ ]:
fig = plt.figure(figsize=(24, 10))
font_settings = {
    "font.size": 24,
    "axes.titlesize": 28,
    "axes.labelsize": 24,
    "legend.fontsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
}
plt.rcParams.update(font_settings)


# Create axes with width ratios: left panel smaller, right panel bigger
gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 1], wspace=0.05)
axes = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]

# Left plot - time series
ax0 = axes[0]
yvals = burned_forest_tseries * 100 / forest_area
ax0.plot(yvals["year"], yvals, "-", color="#B12121", linewidth=6, markersize=15, label="MTBS")
ax0.plot(
    yvals["year"],
    yvals.rolling(year=10, center=True).mean(),
    "--",
    color="#B12121",
    linewidth=6,
    label="MTBS (centered 10-year rolling mean)",
)
ax0.axhline(y=0, linestyle=":", color="k")
ax0.set_ylabel("Burned area (percent of forest area per year)", labelpad=10)
ax0.set_xlabel("Year", labelpad=10)
ax0.spines[["right", "top"]].set_visible(True)
ax0.set_title("Forest area burned annually", pad=15)
ax0.plot(
    fia_frac_plots["year"],
    fia_frac_plots.where(fia_frac_plots["year"] >= 2000),
    color="#E55713",
    linewidth=6,
    label="FIA-derived",
)
ax0.legend()
ax0.text(
    -0.08,
    1.05,
    "a",
    transform=ax0.transAxes,
    fontsize=font_settings["axes.titlesize"] * 1.25,
    fontweight="bold",
    va="top",
)
ax0.tick_params(axis="y", which="both", left=True, right=True, direction="in")
ax0.tick_params(axis="x", which="both", top=True, bottom=True, direction="in")
ax0.legend(frameon=False)

# Right plot - maps
ax1 = axes[1]
maps.plot_map(
    is_forest_clipped,
    shp=western_states,
    latlon=False,
    cbar_label="Percent Forest",
    cmap=plt.cm.binary,
    add_colorbar=False,
    clims=[0, 600],
    savefig=None,
    ax=ax1,
)
maps.plot_map(
    mapdata_clipped,
    shp=western_states,
    latlon=False,
    title="Year of most recent fire",
    cbar_label="Year of most recent fire",
    cmap=discrete_cmap,
    savefig=None,
    clims=[2000, 2025],
    add_colorbar=True,
    ax=ax1,
)
ax1.set_title("Year of most recent fire", pad=15)

plt.rcParams.update(font_settings)

# The colorbar is on a separate axes appended to the figure
for ax in fig.axes:
    if ax != ax0 and ax != ax1:  # it's the colorbar axes
        ax.yaxis.label.set_fontsize(font_settings["axes.titlesize"])
        ax.tick_params(labelsize=font_settings["xtick.labelsize"])
        ax.set_ylabel("Year of most recent fire", labelpad=10)

ax1.text(
    0,
    1.05,
    "b",
    transform=ax1.transAxes,
    fontsize=font_settings["axes.titlesize"] * 1.25,
    fontweight="bold",
    va="top",
)
fig.subplots_adjust(left=0.05, right=0.92, top=0.92, bottom=0.08, wspace=-0.1)
fig.savefig(dir_info.dir_figures + "Figure1.jpg", bbox_inches="tight", dpi=300)
# fig.savefig(dir_info.dir_figures + "Figure1.pdf", bbox_inches="tight", dpi=300)